
# C6-pytorch — Review

Work through this notebook *after* the three lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: a concept summary table, the module and
counting idiom sheet, a 13-item self-quiz, and pointers on what to
redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.


In [ ]:

import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention



## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Torch tensors | NumPy arrays with a torch passport: same shapes, indexing, broadcasting; `dim=` for `axis=` | Float default is **float32** (course resets to float64); ints are int64; `.item()` unwraps 0-d tensors; `torch.manual_seed(20260804)` in the same cell as every reproducible draw |
| Python inheritance | `class Sub(Base)` reuses the base, overriding only what differs | `super().__init__(...)` first, before own setup — the base class's methods depend on the attributes it sets; method lookup starts at the object's own class |
| `nn.Module` | The base class every network inherits from: you supply `forward`, it supplies the machinery | Call modules as `m(x)` — the inherited `__call__` dispatches to your `forward`; no `forward` ⇒ `NotImplementedError` at call time |
| Custom layers | `DenseLayer`: weight `(out, in)`, bias `(out,)`, forward `x @ weight.T + bias`; `ThresholdGate`: `(x >= 0).to(x.dtype)` | One row of `weight` per output unit; gates own **no** parameters; parameter-free modules need no `__init__` |
| `nn.Parameter` / `requires_grad` | Registration makes numbers visible to `parameters()`/`state_dict`; the flag records intent | Plain tensor attributes compute but *vanish* from inspection; `nn.Parameter` defaults the flag to `True`; the course constructs with `requires_grad=False` (hand-set, fixed) |
| Manual weights | The weights are the program: choose them so the module meets a written spec | Half-plane scores feed gates; AND = sum of fired units $- (k - 0.5)$; OR = sum $- 0.5$; a bias vector alone can move a region |
| Parameter counting | Audit a model by hand, by `numel`, and by shapes when `numel` is banned | Dense $\text{in} \to \text{out}$: $\text{out} \times (\text{in} + 1)$; stack: $\sum n_\ell (n_{\ell-1} + 1)$; equal counts ≠ equal shapes |

## Idiom sheet

**The pinned modules (Session 2):**

```python
class DenseLayer(nn.Module):
    def __init__(self, weight, bias):          # weight (out, in), bias (out,)
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)
    def forward(self, x):
        return x @ self.weight.T + self.bias

class ThresholdGate(nn.Module):
    def forward(self, x):
        return (x >= 0).to(x.dtype)
```

**Conventions:** `torch.set_default_dtype(torch.float64)` at the top
of every notebook; `torch.manual_seed(20260804)` beside every
reproducible draw; explicit casts at every NumPy border
(`torch.from_numpy(a).to(torch.float64)`, `.numpy()` both sides
float64); deliberate float32 states `atol=1e-6`/`rtol=1e-5`.

**Inspection:**

```python
[(n, tuple(p.shape)) for n, p in m.named_parameters()]   # names + shapes
sum(p.numel() for p in m.parameters())                   # scalar count
all(not p.requires_grad for p in m.parameters())         # frozen audit
list(m.state_dict().keys())                              # save/load names
m.load_state_dict(new_dict)                              # swap manual weights
```

**Counting (hand register):** dense layer =
$\text{out} \times (\text{in} + 1)$; gates contribute $0$; under a
"no `numel`" ban, multiply out `p.shape` — `reshape(-1)`, `flatten`,
`len`, `np.prod` are the same helper in disguise and score zero.

**Errors worth recognizing on sight:**
`cannot assign parameters before Module.__init__() call` (forgot
`super().__init__()`);
`Module [X] is missing the required "forward" function` (misspelled
`forward`);
`expected m1 and m2 to have the same dtype` (float32 meets float64 —
cast at the border);
`mean(): could not infer output dtype ... Long` (integer tensor asked
for a float reduction).



## Self-quiz (13 items)

Answer everything before opening the collapsed answers at the very
end.

**Q1.** Fresh interpreter, no defaults changed: the dtypes of
`torch.tensor([1.5])`, `torch.tensor([1])`, and `np.array([1.5])` —
and the course's one-line fix for the mismatch this creates.

**Q2.** `t = torch.arange(8).reshape(2, 4)`: give `t.sum(dim=0)`,
`t.sum(dim=1)`, and the shape of each.

**Q3.** `arr = np.zeros(3)`; `t = torch.from_numpy(arr)`;
`arr[2] = 9.0`. What does `t` print, and what one word explains it?

**Q4.** A subclass `__init__` skips `super().__init__()`.
In plain Python, when does the bug surface?
In an `nn.Module` subclass that assigns a parameter, when?

**Q5.** `twice` is defined only in a base class as
`self.apply(self.apply(x))`; a subclass overrides `apply`.
Whose `apply` runs inside an inherited `twice` call, and which rule
decides?

**Q6.** Why does the course call modules as `m(x)` when
`m.forward(x)` returns the same values in this unit?

**Q7.** `class G(nn.Module)` defines `froward` (a typo).
What happens at `G()`, and what at `G()(x)`?

**Q8.** State the pinned `DenseLayer` spec: both parameter shapes for
$\text{in} = 3, \text{out} = 2$, the forward formula, and the value of
`DenseLayer(torch.zeros(2, 3), torch.tensor([1.0, -1.0]))(torch.ones(1, 3))`.

**Q9.** A layer stores `self.weight = torch.as_tensor(W)`.
Name every inspection surface this breaks and the one-line fix.

**Q10.** Fill the table: `requires_grad` of `nn.Parameter(t)`, of
`nn.Parameter(t, requires_grad=False)`, of a plain tensor — and which
one is the course's construction for module numbers.

**Q11.** True or false, with one sentence each: (i) setting
`requires_grad=False` changes a module's forward outputs; (ii) running
10,000 forward passes can drift a parameter whose flag is `True`.

**Q12.** Design manual weights: a `DenseLayer` + `ThresholdGate` pair
(one output unit) that fires exactly when $x_1 \ge 2$, inputs
$(x_1, x_2)$.

**Q13.** Count $3 \to 5 \to 2$ (dense, gate, dense, gate) by hand;
then find the width $h$ for which $2 \to h \to 1$ (same pattern) holds
exactly $33$ parameters.

## What to redo, per weak spot

- **Tensor mechanics, dtypes, seeding (Q1–Q3):** redo p01, p05, p19;
  reread Session 1 §§1–4 and Pitfalls I.
- **Classes, inheritance, `super()` (Q4, Q5):** redo p02, p06;
  Session 1 §§5–6; then p17 as the stretch.
- **`nn.Module` anatomy (Q6, Q7):** redo p07, p20; Session 1 §7 and
  Session 2 §2.
- **The pinned layers and registration (Q8, Q9):** redo p08, p13,
  p14; Session 2 §§3, 5, and Pitfalls II.
- **The flag (Q10, Q11):** redo p03, p16; Session 2 §4.
- **Manual-weight design (Q12):** redo p09, p10, p15, p18; Session 2
  §5.
- **Counting and inspection (Q13):** redo p04, p11, p12; Session 3
  throughout — especially the count-without-`numel` register (§3).
- **The end-to-end texture:** if any hesitation remains, rebuild p13
  from a blank cell — it is the unit in one problem.

## Answers (open only when done)

<details><summary><b>Answers to all 13 quiz items</b></summary>

**A1.** `torch.float32`, `torch.int64`, `float64`.
Fix: `torch.set_default_dtype(torch.float64)` at the top of the
notebook (ints stay int64 — only float defaults move).

**A2.** `t.sum(dim=0)` is `tensor([ 4,  6,  8, 10])`, shape `(4,)`;
`t.sum(dim=1)` is `tensor([ 6, 22])`, shape `(2,)`.

**A3.** `tensor([0., 0., 9.])` — *sharing*: `from_numpy` wraps the
same memory, so writes through either name are visible to both.

**A4.** Plain Python: only when some *inherited* method first touches
a base-class attribute that was never set (`AttributeError`, far from
the cause).
`nn.Module`: immediately at the first parameter assignment —
`AttributeError: cannot assign parameters before Module.__init__()
call`.

**A5.** The subclass's `apply`: method lookup starts at the object's
own class and falls back to the base only when the name is absent.
The same rule routes `nn.Module.__call__` to your `forward`.

**A6.** Only the call form passes through the base class's `__call__`
machinery; later units rely on what that machinery does, so the habit
is part of the contract (and it is the form every torch codebase and
the exam use).

**A7.** `G()` succeeds — nothing checks method names at construction.
`G()(x)` raises `NotImplementedError: Module [G] is missing the
required "forward" function`.

**A8.** Weight `(2, 3)`, bias `(2,)`;
`forward(x) = x @ weight.T + bias`; with zero weights the output is
the broadcast bias: `tensor([[ 1., -1.]])`.

**A9.** Broken: `parameters()`, `named_parameters()`,
`state_dict()` (and with them counting, saving, loading, and the
frozen-flag audit).
Fix: `self.weight = nn.Parameter(torch.as_tensor(W),
requires_grad=False)`.

**A10.** `True`; `False`; `False` — and the middle one,
`nn.Parameter(t, requires_grad=False)`, is the course's construction:
registered *and* marked hand-set.

**A11.** (i) False — the flag is bookkeeping; the arithmetic is
identical.
(ii) False — forward passes only *read* parameters; nothing in this
course's use of torch ever writes them (p16's experiment is the
evidence).

**A12.** `DenseLayer(torch.tensor([[1.0, 0.0]]), torch.tensor([-2.0]))`
into a `ThresholdGate`: score $x_1 - 2 \ge 0$ exactly when
$x_1 \ge 2$; the $x_2$ weight is $0$.

**A13.** $5 \times (3 + 1) + 2 \times (5 + 1) = 20 + 12 = 32$.
For $2 \to h \to 1$: $P(h) = h(2 + 1) + (h + 1) = 4h + 1 = 33$
gives $h = 8$.

</details>
